# Experimental Training Pipeline - GBC Biodata Inventory

**Purpose**: Systematic hyperparameter tuning and experimental training

**Author**: GBC Biodata Inventory Team  
**Created**: 2025-10-29  
**Environment**: Google Colab with GPU

## Overview

This notebook runs multiple training experiments with different hyperparameter configurations:
- Learning rate variations
- Dropout configurations
- Early stopping
- Batch size optimization

## Features

- **TEST_MODE**: Quick validation with minimal epochs
- **Experiment Tracking**: Automatic logging of all configurations and results
- **Session Archives**: Complete session history with unique IDs
- **Comparison Reports**: Automated result comparison and best model selection
- **Google Drive Integration**: Persistent storage and checkpointing

## Session Management

Each experimental session gets a unique ID: `YYYY-MM-DD-abcdef`
All outputs are saved to:
- Local: `experiments/{session_id}/`
- Drive: `MyDrive/inventory_experiments/{session_id}/`

## Cell 1: Mount Google Drive and Setup Session

In [ ]:
# Mount Google Drive
from google.colab import drive
import os
import datetime
import random
import string

# Mount drive
drive.mount('/content/drive', force_remount=True)

# Set base paths
PROJECT_NAME = "inventory_2022"
DRIVE_BASE = f"/content/drive/MyDrive/{PROJECT_NAME}"

# Change to project directory
os.chdir(DRIVE_BASE)

print(f"✅ Mounted Google Drive")
print(f"📁 Working directory: {os.getcwd()}")

# Generate unique session ID: YYYY-MM-DD-abcdef
date_str = datetime.datetime.now().strftime("%Y-%m-%d")
random_suffix = ''.join(random.choices(string.ascii_lowercase + string.digits, k=6))
SESSION_ID = f"{date_str}-{random_suffix}"

print(f"\n🔬 Session ID: {SESSION_ID}")

# Create experiment directories
EXPERIMENT_DIR = f"experiments/{SESSION_ID}"
ARCHIVE_DIR = f"{DRIVE_BASE}/experiment_archives/{SESSION_ID}"

os.makedirs(EXPERIMENT_DIR, exist_ok=True)
os.makedirs(ARCHIVE_DIR, exist_ok=True)

print(f"📂 Experiment directory: {EXPERIMENT_DIR}")
print(f"📦 Archive directory: {ARCHIVE_DIR}")

## Cell 2: Configuration and Experiment Setup

In [ ]:
# ========================================
# CONFIGURATION - V2 BASELINE REPRODUCTION (2025-10-30)
# ========================================

# TEST MODE: Set to True for quick validation (1-2 epochs)
TEST_MODE = False  # Set to True for testing, False for full experiments

# ========================================
# V2 BASELINE CONFIGURATION (FROM SNAKEMAKE)
# ========================================
# Source: config/train_predict.yml, config/models_info.tsv
# V2 Production Results: Classification F1=0.898, NER F1=0.749
# Training Date: October 21, 2025
#
# CRITICAL FINDINGS (2025-10-30):
# - Previous experiments used WRONG learning rates (1e-5, 5e-5)
# - V2 uses 2e-5 for BOTH classification and NER
# - V2 uses batch_size=16 (not 32)
# - V2 uses NO early stopping (trains full 10 epochs)
# - V2 uses NO learning rate scheduler
# ========================================

BASE_CONFIG = {
    'batch_size': 16,              # V2 exact value (not 32)
    'num_epochs': 2 if TEST_MODE else 10,  # V2 exact value (not 15)
    'early_stopping': False,       # V2 doesn't use it
    'patience': None,              # Not used
    'test_mode': TEST_MODE
}

# ========================================
# V2 BASELINE REPRODUCTION EXPERIMENT
# ========================================
# Goal: Match V2 baseline performance to validate configuration
# Expected Results: Classification F1≈0.898, NER F1≈0.749
# ========================================

if TEST_MODE:
    print("⚡ TEST MODE ENABLED - Quick validation with minimal epochs")
    MODEL_EXPERIMENTS = [
        {
            'name': 'v2_baseline_test',
            'model_name': 'allenai/dsp_roberta_base_dapt_biomed_tapt_rct_500',
            'description': 'V2 baseline test run',
            'classif_lr': 2e-5,  # V2 EXACT VALUE
            'ner_lr': 2e-5,      # V2 EXACT VALUE
            'experiment_name': 'exp1_v2_baseline_test'
        }
    ]
else:
    print("🚀 PHASE 0: V2 BASELINE REPRODUCTION")
    MODEL_EXPERIMENTS = [
        # Experiment 1: V2 EXACT REPRODUCTION
        {
            'name': 'v2_exact_reproduction',
            'model_name': 'allenai/dsp_roberta_base_dapt_biomed_tapt_rct_500',
            'description': 'V2 baseline exact reproduction with 2e-5 LR',
            'classif_lr': 2e-5,  # V2 EXACT VALUE (was 1e-5 - WRONG)
            'ner_lr': 2e-5,      # V2 EXACT VALUE (was 5e-5 - WRONG)
            'experiment_name': 'exp1_v2_exact'
        },
        
        # Experiment 2: V2 with slightly LOWER LR
        {
            'name': 'v2_lower_lr',
            'model_name': 'allenai/dsp_roberta_base_dapt_biomed_tapt_rct_500',
            'description': 'V2 baseline with slightly lower LR (1.5e-5)',
            'classif_lr': 1.5e-5,
            'ner_lr': 1.5e-5,
            'experiment_name': 'exp2_v2_lower'
        },
        
        # Experiment 3: V2 with slightly HIGHER LR
        {
            'name': 'v2_higher_lr',
            'model_name': 'allenai/dsp_roberta_base_dapt_biomed_tapt_rct_500',
            'description': 'V2 baseline with slightly higher LR (3e-5)',
            'classif_lr': 3e-5,
            'ner_lr': 3e-5,
            'experiment_name': 'exp3_v2_higher'
        },
        
        # Experiment 4: Test ASYMMETRIC LRs (different for classif vs NER)
        {
            'name': 'asymmetric_lr',
            'model_name': 'allenai/dsp_roberta_base_dapt_biomed_tapt_rct_500',
            'description': 'Test different LRs: classif=2e-5, ner=1.5e-5',
            'classif_lr': 2e-5,
            'ner_lr': 1.5e-5,
            'experiment_name': 'exp4_asymmetric'
        }
    ]

print(f"\n📊 Experiments configured: {len(MODEL_EXPERIMENTS)}")
print(f"\n{'='*80}")
print("EXPERIMENT PLAN - V2 BASELINE REPRODUCTION")
print(f"{'='*80}")

for i, config in enumerate(MODEL_EXPERIMENTS, 1):
    print(f"\n{i}. {config['name']}")
    print(f"   Model: {config['model_name']}")
    print(f"   Description: {config['description']}")
    print(f"   LR: classif={config['classif_lr']}, ner={config['ner_lr']}")

# Calculate total training time estimate
epochs_per_exp = BASE_CONFIG['num_epochs']
num_experiments = len(MODEL_EXPERIMENTS)
estimated_hours = num_experiments * epochs_per_exp * (0.1 if TEST_MODE else 0.8)

print(f"\n{'='*80}")
print("TRAINING ESTIMATES")
print(f"{'='*80}")
print(f"Total experiments: {num_experiments}")
print(f"Epochs per experiment: {epochs_per_exp}")
print(f"Estimated time per epoch: ~{0.1 if TEST_MODE else 0.8}h")
print(f"Estimated total time: ~{estimated_hours:.1f} hours ({estimated_hours*60:.0f} minutes)")

print(f"\n{'='*80}")
print("KEY CONFIGURATION CHANGES FROM PREVIOUS SESSIONS")
print(f"{'='*80}")
print("✅ CORRECTED: Learning rates set to V2 baseline (2e-5 for both)")
print("   - Previous session used: classif=1e-5 (50% too low), ner=5e-5 (150% too high)")
print("   - V2 actual values: classif=2e-5, ner=2e-5")
print("\n✅ CORRECTED: Batch size set to V2 baseline (16)")
print("   - Previous session used: 32")
print("   - V2 actual value: 16")
print("\n✅ CORRECTED: Early stopping DISABLED")
print("   - Previous session used: early_stopping=True, patience=5")
print("   - V2 configuration: NO early stopping, trains full 10 epochs")
print("\n✅ CORRECTED: Epochs set to V2 baseline (10)")
print("   - Previous session used: 15")
print("   - V2 actual value: 10")
print("\n✅ MAINTAINED: Using FIXED production splits")
print("   - data/classif_splits_full/")
print("   - data/ner_splits_full/")

print(f"\n{'='*80}")
print("SUCCESS CRITERIA")
print(f"{'='*80}")
print("Primary Goal (Experiment 1 - V2 Exact Reproduction):")
print("  - Classification F1 ≥ 0.895 (within 0.3% of V2 baseline 0.898)")
print("  - NER F1 ≥ 0.745 (within 0.5% of V2 baseline 0.749)")
print("  - If matched: Phase 0 COMPLETE ✅")
print("\nSecondary Goals (Experiments 2-4):")
print("  - Identify if small LR variations improve performance")
print("  - Test asymmetric LRs (different for classif vs NER)")
print("  - Gather evidence for Phase 1 optimization")

print(f"\n{'='*80}")
print("EXPECTED OUTCOMES")
print(f"{'='*80}")
print("Exp 1 (2e-5, 2e-5): Should MATCH V2 baseline (0.898, 0.749)")
print("Exp 2 (1.5e-5, 1.5e-5): Likely slower convergence, similar final performance")
print("Exp 3 (3e-5, 3e-5): Potentially faster convergence, risk of instability")
print("Exp 4 (2e-5, 1.5e-5): Test if NER benefits from lower LR")

print(f"\n{'='*80}")
print("SOURCE DOCUMENTATION")
print(f"{'='*80}")
print("V2 Configuration Source: snakemake/config/train_predict.yml")
print("V2 Model Parameters: snakemake/config/models_info.tsv")
print("V2 Training Archive: trained_models_25/2025-10-21_full_production_training/")
print("Analysis Document: docs/V2_BASELINE_PARAMETERS_FOUND.md")
print("Clarification Document: docs/PARAMETER_OPTIMIZATION_CLARIFICATION.md")

## Cell 3: Environment Setup and GPU Optimization

In [ ]:
import sys
import subprocess
import torch

# Add src to path
if '/content/drive/MyDrive/inventory_2022/src' not in sys.path:
    sys.path.insert(0, '/content/drive/MyDrive/inventory_2022/src')

print("📦 Installing/upgrading dependencies...")
# Pin transformers to compatible version (4.35.2) to avoid PyTorch compatibility issues
# Do NOT upgrade torch - use Colab's default PyTorch version
subprocess.run(['pip', 'install', '-q', 'transformers==4.35.2', 'datasets', 'evaluate', 'seqeval',
                'scikit-learn', 'matplotlib', 'seaborn', 'pandas', 'nltk'],
               check=False)

# Download NLTK data
print("\n📥 Downloading NLTK data...")
import nltk
import ssl

try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

nltk.download('punkt_tab')
print("✅ NLTK data downloaded")

print("\n🔧 Importing modules...")
from src.experimental_utils import (
    EarlyStopping,
    ExperimentTracker,
    calculate_optimal_batch_size,
    plot_training_curves,
    generate_experiment_report
)
from src.training_utils import (
    clear_gpu_memory,
    get_gpu_memory_info,
    check_prerequisites
)

print("✅ Modules imported successfully")

# GPU Setup
print("\n" + "="*60)
print("GPU CONFIGURATION")
print("="*60)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"✅ GPU Available: {gpu_name}")
    print(f"   Memory: {gpu_memory:.1f} GB")
    
    # V2 uses fixed batch_size=16 (not calculated)
    # BASE_CONFIG already set to 16 in Cell 4
    print(f"   Batch size: {BASE_CONFIG['batch_size']} (V2 baseline value)")
    
    # Clear GPU memory
    clear_gpu_memory()
    get_gpu_memory_info()
else:
    print("⚠️ No GPU available - training will be slow")
    # Keep batch_size=16 even without GPU for consistency

print("\n✅ Environment setup complete")

## Cell 4: Initialize Experiment Tracker

In [ ]:
# Initialize experiment tracker
tracker = ExperimentTracker(session_id=SESSION_ID, output_dir="experiments")

print("🔬 Experiment Tracker Initialized")
print(f"   Session: {SESSION_ID}")
print(f"   Output: {EXPERIMENT_DIR}")
print(f"   Archive: {ARCHIVE_DIR}")
print(f"\n📝 Ready to track {len(MODEL_EXPERIMENTS)} experiments")

## Cell 5: Data Preparation and Prerequisites Check

In [ ]:
from pathlib import Path

print("📋 Checking prerequisites...\n")

# Check required data files exist
check_prerequisites()

# ========================================
# CRITICAL FIX (2025-10-29): Use FIXED PRODUCTION SPLITS
# ========================================
# Previously generated session-specific splits, which prevented baseline comparison
# Now using fixed production splits to enable reproducible results
# Respects TEST_MODE for quick validation runs
# ========================================

if TEST_MODE:
    split_dir = "data/classif_splits_test"
    ner_split_dir = "data/ner_splits_test"
    print("⚡ TEST MODE: Using test splits")
else:
    split_dir = "data/classif_splits_full"
    ner_split_dir = "data/ner_splits_full"
    print("🚀 FULL TRAINING: Using production splits")

# Verify splits exist
if not Path(split_dir).exists():
    raise FileNotFoundError(f"Classification splits not found: {split_dir}")
if not Path(ner_split_dir).exists():
    raise FileNotFoundError(f"NER splits not found: {ner_split_dir}")

print(f"✅ Using FIXED splits (enables baseline comparison)")
print(f"   Classification: {split_dir}")
print(f"   NER: {ner_split_dir}")

# Store split directories in config
BASE_CONFIG['classif_split_dir'] = split_dir
BASE_CONFIG['ner_split_dir'] = ner_split_dir

## Cell 5.5: Pre-Flight Checks and Validation

In [ ]:
print("=" * 80)
print("PRE-FLIGHT CHECKS")
print("=" * 80)

checks_passed = True

# Check 1: experimental_utils import
print("\n1. Testing experimental_utils import...")
try:
    from src.experimental_utils import EarlyStopping, ExperimentTracker
    print("   ✅ experimental_utils imported successfully")
except ImportError as e:
    print(f"   ❌ Failed to import experimental_utils: {e}")
    checks_passed = False

# Check 2: training modules import
print("\n2. Testing training modules import...")
try:
    import sys
    if "/content/drive/MyDrive/inventory_2022/src" not in sys.path:
        sys.path.insert(0, "/content/drive/MyDrive/inventory_2022/src")
    import class_train
    import ner_train
    print("   ✅ Training modules imported successfully")
except ImportError as e:
    print(f"   ❌ Failed to import training modules: {e}")
    checks_passed = False

# Check 3: NLTK data
print("\n3. Checking NLTK punkt_tab availability...")
try:
    import nltk
    try:
        nltk.data.find("tokenizers/punkt_tab")
        print("   ✅ NLTK punkt_tab available")
    except LookupError:
        print("   ⚠️ NLTK punkt_tab not found, downloading...")
        nltk.download("punkt_tab", quiet=True)
        print("   ✅ NLTK punkt_tab downloaded")
except Exception as e:
    print(f"   ❌ NLTK check failed: {e}")
    checks_passed = False

# Check 4: model accessibility
print("\n4. Testing model download capability...")
try:
    from transformers import AutoTokenizer
    # Use first model from MODEL_EXPERIMENTS for testing
    test_model = MODEL_EXPERIMENTS[0]["model_name"]
    print(f"   Testing: {test_model}")
    tokenizer = AutoTokenizer.from_pretrained(test_model)
    print(f"   ✅ Model {test_model} accessible")
except Exception as e:
    print(f"   ❌ Model download test failed: {e}")
    checks_passed = False

# Check 5: data files
print("\n5. Verifying training data files...")
try:
    from pathlib import Path
    # Use the split directories set in Cell 10
    required_files = [
        f"{BASE_CONFIG['classif_split_dir']}/train_paper_classif.csv",
        f"{BASE_CONFIG['classif_split_dir']}/val_paper_classif.csv",
        f"{BASE_CONFIG['ner_split_dir']}/train_ner.pkl",
        f"{BASE_CONFIG['ner_split_dir']}/val_ner.pkl"
    ]
    
    missing_files = []
    for filepath in required_files:
        if not Path(filepath).exists():
            missing_files.append(filepath)
    
    if missing_files:
        print(f"   ❌ Missing data files: {missing_files}")
        checks_passed = False
    else:
        print(f"   ✅ All {len(required_files)} data files found")
except Exception as e:
    print(f"   ❌ Data file check failed: {e}")
    checks_passed = False

# Check 6: GPU
print("\n6. Checking GPU availability...")
try:
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"   ✅ GPU available: {gpu_name}")
        print(f"   ✅ GPU memory: {gpu_memory:.1f} GB")
    else:
        print("   ⚠️ No GPU available - training will be slow")
except Exception as e:
    print(f"   ❌ GPU check failed: {e}")
    checks_passed = False

# Summary
print("\n" + "=" * 80)
if checks_passed:
    print("✅ ALL PRE-FLIGHT CHECKS PASSED")
    print("=" * 80)
else:
    print("❌ PRE-FLIGHT CHECKS FAILED")
    print("=" * 80)
    raise RuntimeError("Pre-flight checks failed. Please resolve issues above before continuing.")

## Cell 6: Main Training Loop - Run All Experiments

In [ ]:
import time
import pandas as pd
import subprocess
import traceback
from datetime import datetime
from pathlib import Path

print("="*80)
print(f"STARTING EXPERIMENTAL TRAINING SESSION: {SESSION_ID}")
print("="*80)
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Mode: {'TEST' if TEST_MODE else 'FULL TRAINING - MULTI-MODEL COMPARISON'}")
print(f"Configurations: {len(MODEL_EXPERIMENTS)}")
print("="*80)

# Run experiments sequentially
for exp_idx, model_config in enumerate(MODEL_EXPERIMENTS, 1):
    
    # Create experiment-specific configuration
    exp_config = BASE_CONFIG.copy()
    exp_config.update(model_config)
    exp_name = f"exp{exp_idx}_{model_config['name']}"
    exp_config['experiment_name'] = exp_name
    
    # Generate experiment ID
    exp_id = f"{SESSION_ID}_{exp_name}"
    
    # Start tracking this experiment
    tracker.start_experiment(exp_config)
    
    exp_start_time = time.time()
    
    try:
        # Create experiment-specific output directories
        exp_dir = Path(EXPERIMENT_DIR) / exp_name
        exp_dir.mkdir(exist_ok=True)
        
        classif_output = f"out/classif_{SESSION_ID}_{exp_name}"
        ner_output = f"out/ner_{SESSION_ID}_{exp_name}"
        
        os.makedirs(classif_output, exist_ok=True)
        os.makedirs(ner_output, exist_ok=True)
        
        # CREATE LOGS DIRECTORY
        logs_dir = exp_dir / "training_logs"
        logs_dir.mkdir(exist_ok=True)
        
        print(f"\n{'='*80}")
        print(f"EXPERIMENT {exp_idx}/{len(MODEL_EXPERIMENTS)}: {model_config['name']}")
        print(f"{'='*80}")
        print(f"Model: {model_config['model_name']}")
        if 'description' in model_config:
            print(f"Info: {model_config['description']}")
        
        # ========================================
        # CLASSIFICATION TRAINING WITH LOGGING
        # ========================================
        print(f"\n🔵 CLASSIFICATION TRAINING")
        print(f"   Learning rate: {exp_config['classif_lr']}")
        print(f"   Output: {classif_output}")
        
        classif_log = logs_dir / f"{exp_name}_classification.log"
        
        classif_cmd = [
            'python', 'src/class_train.py',
            '-t', f"{exp_config['classif_split_dir']}/train_paper_classif.csv",
            '-v', f"{exp_config['classif_split_dir']}/val_paper_classif.csv",
            '-o', classif_output,
            '-m', exp_config['model_name'],
            '-rate', str(exp_config['classif_lr']),
            '-ne', str(exp_config['num_epochs']),
            '-batch', str(exp_config['batch_size']),
            '-r'
        ]
        
        if exp_config['early_stopping']:
            classif_cmd.extend(['--early-stopping', '--patience', str(exp_config['patience'])])
        
        # Run with output capture
        print("   Running classification training...")
        result = subprocess.run(
            classif_cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            timeout=7200  # 2 hours timeout (increased for longer training)
        )
        
        # Save complete output to log file
        with open(classif_log, 'w') as f:
            f.write(result.stdout)
        
        # Display last 50 lines in notebook
        log_lines = result.stdout.split('\n')
        print("\n   Last 50 lines of output:")
        print('\n'.join(log_lines[-50:]))
        
        if result.returncode != 0:
            raise RuntimeError(f"Classification training failed with code {result.returncode}")
        
        # Load classification results
        classif_stats = pd.read_csv(f"{classif_output}/train_stats.csv")
        best_classif_idx = classif_stats['val_f1'].idxmax()
        best_classif = classif_stats.loc[best_classif_idx]
        
        classif_results = {
            'val_f1': best_classif['val_f1'],
            'val_precision': best_classif['val_precision'],
            'val_recall': best_classif['val_recall'],
            'train_f1': best_classif['train_f1'],
            'best_epoch': int(best_classif['epoch']),
            'total_epochs': len(classif_stats)
        }
        
        print(f"\n✅ Classification complete: Val F1 = {classif_results['val_f1']:.4f}")
        print(f"   Best epoch: {classif_results['best_epoch']}/{classif_results['total_epochs']}")
        print(f"   Log saved to: {classif_log}")
        
        # Clear GPU memory
        clear_gpu_memory()
        
        # ========================================
        # NER TRAINING WITH LOGGING
        # ========================================
        print(f"\n🟢 NER TRAINING")
        print(f"   Learning rate: {exp_config['ner_lr']}")
        print(f"   Output: {ner_output}")
        
        ner_log = logs_dir / f"{exp_name}_ner.log"
        
        ner_cmd = [
            'python', 'src/ner_train.py',
            '-t', f"{exp_config['ner_split_dir']}/train_ner.pkl",
            '-v', f"{exp_config['ner_split_dir']}/val_ner.pkl",
            '-o', ner_output,
            '-m', exp_config['model_name'],
            '-rate', str(exp_config['ner_lr']),
            '-ne', str(exp_config['num_epochs']),
            '-batch', str(exp_config['batch_size']),
            '-r'
        ]
        
        if exp_config['early_stopping']:
            ner_cmd.extend(['--early-stopping', '--patience', str(exp_config['patience'])])
        
        # Run with output capture
        print("   Running NER training...")
        result = subprocess.run(
            ner_cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            timeout=7200  # 2 hours timeout (increased for longer training)
        )
        
        # Save complete output to log file
        with open(ner_log, 'w') as f:
            f.write(result.stdout)
        
        # Display last 50 lines in notebook
        log_lines = result.stdout.split('\n')
        print("\n   Last 50 lines of output:")
        print('\n'.join(log_lines[-50:]))
        
        if result.returncode != 0:
            raise RuntimeError(f"NER training failed with code {result.returncode}")
        
        # Load NER results
        ner_stats = pd.read_csv(f"{ner_output}/train_stats.csv")
        best_ner_idx = ner_stats['val_f1'].idxmax()
        best_ner = ner_stats.loc[best_ner_idx]
        
        ner_results = {
            'val_f1': best_ner['val_f1'],
            'val_precision': best_ner['val_precision'],
            'val_recall': best_ner['val_recall'],
            'train_f1': best_ner['train_f1'],
            'best_epoch': int(best_ner['epoch']),
            'total_epochs': len(ner_stats)
        }
        
        print(f"\n✅ NER complete: Val F1 = {ner_results['val_f1']:.4f}")
        print(f"   Best epoch: {ner_results['best_epoch']}/{ner_results['total_epochs']}")
        print(f"   Log saved to: {ner_log}")
        
        # Clear GPU memory
        clear_gpu_memory()
        
        # Record experiment results
        exp_time = time.time() - exp_start_time
        tracker.record_results(classif_results, ner_results, exp_time)
        
        # Generate plots for this experiment
        plot_training_curves(
            classif_stats,
            f"{EXPERIMENT_DIR}/{exp_name}_classif_curves.png",
            f"Classification Training - {model_config['name']}"
        )
        
        plot_training_curves(
            ner_stats,
            f"{EXPERIMENT_DIR}/{exp_name}_ner_curves.png",
            f"NER Training - {model_config['name']}"
        )
        
        print(f"\n✅ Experiment {exp_idx} completed in {exp_time/60:.1f} minutes")
        
    except subprocess.TimeoutExpired as e:
        print(f"\n❌ Experiment {exp_idx} TIMEOUT: {e.timeout}s")
        full_traceback = traceback.format_exc()
        print(full_traceback)
        
        tracker.record_experiment_failure(
            exp_id,
            exp_config,
            f"Timeout after {e.timeout}s",
            full_traceback
        )
        continue
        
    except Exception as e:
        print(f"\n❌ Experiment {exp_idx} FAILED: {str(e)}")
        full_traceback = traceback.format_exc()
        print(full_traceback)
        
        # Find log file with content
        log_file = None
        if classif_log.exists() and classif_log.stat().st_size > 0:
            log_file = classif_log
        elif ner_log.exists() and ner_log.stat().st_size > 0:
            log_file = ner_log
        
        tracker.record_experiment_failure(
            exp_id,
            exp_config,
            str(e),
            full_traceback,
            log_file
        )
        continue

print("\n" + "="*80)
print("ALL EXPERIMENTS COMPLETED")
print("="*80)
print(f"Finished: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## Cell 7: Results Analysis and Visualization

In [ ]:
print("="*60)
print("RESULTS ANALYSIS")
print("="*60)

# Save experiment results
results_file = tracker.save_results()
print(f"\n📊 Results saved: {results_file}")

# Generate comparison summary
summary_file = tracker.generate_comparison_summary()
print(f"📝 Summary saved: {summary_file}")

# Display results DataFrame
results_df = pd.read_csv(results_file)
print("\n📈 EXPERIMENT RESULTS SUMMARY:\n")
display(results_df)

# Find best configurations
completed = results_df[results_df['status'] == 'completed']

if len(completed) > 0:
    best_classif_idx = completed['classif_val_f1'].idxmax()
    best_ner_idx = completed['ner_val_f1'].idxmax()
    
    print("\n" + "="*60)
    print("BEST CONFIGURATIONS")
    print("="*60)
    
    print("\n🏆 BEST CLASSIFICATION MODEL:")
    best_classif = completed.loc[best_classif_idx]
    print(f"   Config: {best_classif['config_name']}")
    print(f"   Learning Rate: {best_classif['config_classif_lr']}")
    print(f"   Val F1: {best_classif['classif_val_f1']:.4f}")
    print(f"   Best Epoch: {best_classif['classif_best_epoch']}")
    
    print("\n🏆 BEST NER MODEL:")
    best_ner = completed.loc[best_ner_idx]
    print(f"   Config: {best_ner['config_name']}")
    print(f"   Learning Rate: {best_ner['config_ner_lr']}")
    print(f"   Val F1: {best_ner['ner_val_f1']:.4f}")
    print(f"   Best Epoch: {best_ner['ner_best_epoch']}")

# Display training curves summary
print("\n📊 Training curves saved to:")
for png_file in Path(EXPERIMENT_DIR).glob("*.png"):
    print(f"   {png_file}")

print("\n✅ Analysis complete")

## Cell 8: Archive Session and Cleanup

In [ ]:
import shutil

print("="*60)
print("SESSION ARCHIVAL")
print("="*60)

# Create comprehensive archive
print("\n📦 Creating session archive...")
archive_path = tracker.create_session_archive()

# Copy experiment outputs to archive
print("\n📁 Copying experiment outputs...")
for exp_dir in Path("out").glob(f"*exp_{SESSION_ID}*"):
    dest = Path(archive_path) / exp_dir.name
    shutil.copytree(exp_dir, dest, dirs_exist_ok=True)
    print(f"   ✅ {exp_dir.name}")

# Copy training curves
curves_dir = Path(archive_path) / "training_curves"
curves_dir.mkdir(exist_ok=True)
for png_file in Path(EXPERIMENT_DIR).glob("*.png"):
    shutil.copy2(png_file, curves_dir / png_file.name)

# Archive training logs
print("\n📝 Archiving training logs...")
for exp_subdir in Path(EXPERIMENT_DIR).glob("exp*"):
    if exp_subdir.is_dir():
        logs_subdir = exp_subdir / "training_logs"
        if logs_subdir.exists():
            dest_logs = Path(archive_path) / "training_logs" / exp_subdir.name
            shutil.copytree(logs_subdir, dest_logs, dirs_exist_ok=True)
            print(f"   ✅ {exp_subdir.name}/training_logs")

# Archive error details
print("\n📝 Archiving error details...")
error_files = list(Path(EXPERIMENT_DIR).glob("*_error_details.txt"))
if error_files:
    error_dest = Path(archive_path) / "error_details"
    error_dest.mkdir(exist_ok=True)
    for error_file in error_files:
        shutil.copy2(error_file, error_dest / error_file.name)
        print(f"   ✅ {error_file.name}")
else:
    print("   ℹ️ No error details to archive")

print("\n✅ Logs and error details archived")

    
print(f"\n✅ Session archived to: {archive_path}")

# Display archive contents
print("\n📂 Archive contents:")
for item in Path(archive_path).rglob("*"):
    if item.is_file():
        size_mb = item.stat().st_size / (1024*1024)
        rel_path = item.relative_to(archive_path)
        if size_mb > 0.1:  # Only show files > 100KB
            print(f"   {rel_path} ({size_mb:.1f} MB)")

# Summary statistics
total_size = sum(f.stat().st_size for f in Path(archive_path).rglob('*') if f.is_file())
total_size_mb = total_size / (1024*1024)

print("\n" + "="*60)
print("SESSION SUMMARY")
print("="*60)
print(f"Session ID: {SESSION_ID}")
print(f"Mode: {'TEST' if TEST_MODE else 'FULL TRAINING'}")
print(f"Experiments Run: {len(tracker.experiments)}")
print(f"Experiments Completed: {len([e for e in tracker.experiments if e['status'] == 'completed'])}")
print(f"Experiments Failed: {len([e for e in tracker.experiments if e['status'] == 'failed'])}")
print(f"Archive Size: {total_size_mb:.1f} MB")
print(f"Archive Location: {archive_path}")

print("\n🎉 EXPERIMENTAL SESSION COMPLETE!")
print("\n📝 Next steps:")
print("   1. Review comparison_summary.md for best configurations")
print("   2. Examine training curves in training_curves/")
print("   3. Use best model for production deployment")
print("   4. Document findings in experiment log")